# Interactive Optimization with ADM and Pymoo-based Problem Server

This notebook demonstrates how to use an ADM framework to interactively solve benchmark problems (DTLZ, WFG) exposed through a FastAPI problem server.

## 1. Introduction

This notebook demonstrates how to connect to a FastAPI-based *Problem Server* that exposes
multi-objective benchmark problems (e.g., DTLZ and WFG families) through a REST API.

⚠️ **Important:** Before running this notebook, make sure the problem server (`pymoo_server.py`) is running locally.

You can start it by opening a terminal and running:

```bash
python pymoo_server.py
```

The server should be accessible at `http://127.0.0.1:8000`.

Once the server is up, this notebook will:

1. Connect to it to create problem instances via `server_problem()`.
2. Dynamically generate DTLZ and WFG problems with customizable numbers of objectives and variables.
3. Solve them using the Artificial Decision Maker (ADM) framework and interactive evolutionary algorithms (NSGA-III and RVEA).
4. Visualize the ADM’s learning and decision phases.

## 2. Imports and setup


In [11]:
import numpy as np
import pandas as pd
from desdeo.problem import Problem
import numpy as np
from desdeo.emo.methods.EAs import nsga3, rvea
from desdeo.adm.ADMAfsar import ADMAfsar
from desdeo.emo.hooks.archivers import NonDominatedArchive
from desdeo.problem.testproblems.benchmarks_server import PymooParameters, server_problem
import time
import os

# -- Configuration: Server URL and port
SERVER_URL = "http://127.0.0.1"
SERVER_PORT = 8000

OUTPUT_DIR = "results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Ready to connect to Problem Server at:", f"{SERVER_URL}:{SERVER_PORT}")

✅ Ready to connect to Problem Server at: http://127.0.0.1:8000


## 3. Define Helper Functions for DTLZ and WFG Problems

DTLZ and WFG problems have different default numbers of variables depending on the number of objectives.
These helper functions encapsulate that logic.

In [12]:
def get_dtlz_default_nvar(problem_name: str, n_obj: int) -> int:
    """
    Get the recommended number of variables for a given DTLZ problem and number of objectives.
    Source: pymoo defaults.
    """
    base = {
        "dtlz1": 5,
        "dtlz2": 10,
        "dtlz3": 10,
        "dtlz4": 10,
        "dtlz5": 10,
        "dtlz6": 10,
        "dtlz7": 20,
    }
    k = base.get(problem_name, 10)
    return n_obj + k - 1


def get_wfg_default_nvar(problem_name: str, n_obj: int) -> int:
    """
    WFG problems have n_var = k + l where typically:
      k = 2 * (n_obj - 1)
      l = 20
    """
    k = 2 * (n_obj - 1)
    l = 20
    return k + l

## 4. Problem Creation via the Server


In [ ]:
def get_default_ea_parameters(problem_name: str, n_obj: int):
    """
    Returns the default population size and number of generations
    for NSGA-III / RVEA depending on the problem type and number of objectives.

    Parameters
    ----------
    problem_name : str
        Name of the problem (e.g. "dtlz1", "wfg2", etc.)
    n_obj : int
        Number of objectives.

    Returns
    -------
    pop_size : int
        Suggested population size.
    n_gen : int
        Suggested number of generations.
    """
    problem_name = problem_name.lower()

    # Default templates
    dtlz_defaults = {
        3:  (100, 250),
        5:  (210, 350),
        7:  (330, 400),
        9:  (450, 500)
    }

    wfg_defaults = {
        3:  (120, 400),
        5:  (210, 500),
        7:  (330, 600),
        9:  (450, 800)
    }

    # Choose based on problem family
    if problem_name.startswith("dtlz"):
        pop_size, n_gen = dtlz_defaults.get(n_obj, (100, 300))
    elif problem_name.startswith("wfg"):
        pop_size, n_gen = wfg_defaults.get(n_obj, (120, 500))
    else:
        raise ValueError(f"Unknown problem family: {problem_name}")

    # Special cases for harder DTLZs
    if problem_name in ["dtlz1", "dtlz7"]:
        n_gen = int(n_gen * 1.5)  # need more generations

    return pop_size, n_gen


def get_ideal_nadir(problem_name: str, n_obj: int):
    """
    Returns the ideal and nadir vectors for a DTLZ or WFG problem,
    along with symbols for each objective.

    Parameters
    ----------
    problem_name : str
        Name of the problem, e.g., "dtlz1", "dtlz2", "wfg3", etc.
    n_obj : int
        Number of objectives.

    Returns
    -------
    symbols : list[str]
        Objective symbols, e.g., ["f_1", "f_2", ...].
    ideal_dict : dict
        Mapping from objective symbol to ideal value.
    nadir_dict : dict
        Mapping from objective symbol to nadir value.
    """
    import numpy as np

    symbols = [f"f_{i+1}" for i in range(n_obj)]

    problem_name = problem_name.lower()

    if problem_name.startswith("dtlz"):
        if problem_name == "dtlz1":
            ideal = np.zeros(n_obj)
            nadir = np.full(n_obj, 0.5)
        elif problem_name in ["dtlz2", "dtlz3", "dtlz4", "dtlz5", "dtlz6"]:
            ideal = np.zeros(n_obj)
            nadir = np.ones(n_obj)
        elif problem_name == "dtlz7":
            ideal = np.zeros(n_obj)
            ideal[-1] = 1.0
            nadir = np.ones(n_obj)
            nadir[-1] = 2.0
        else:
            raise ValueError(f"Unknown DTLZ problem: {problem_name}")

    elif problem_name.startswith("wfg"):
        ideal = np.zeros(n_obj)
        # Nadir: 2 * objective index
        nadir = np.array([2*(i+1) for i in range(n_obj)])
    else:
        raise ValueError(f"Unknown problem family: {problem_name}")

    ideal_dict = dict(zip(symbols, ideal))
    nadir_dict = dict(zip(symbols, nadir))

    return ideal_dict, nadir_dict

def create_problem(name: str, n_obj: int):
    """Create a Problem instance from the FastAPI server."""
    family = "dtlz" if name.startswith("dtlz") else "wfg"
    n_var = get_dtlz_default_nvar(name, n_obj) if family == "dtlz" else get_wfg_default_nvar(name, n_obj)
    params = PymooParameters(name=name, n_var=n_var, n_obj=n_obj)
    problem = server_problem(params)
    ideal, nadir = get_ideal_nadir(name, n_obj)
    problem = problem.update_ideal_and_nadir(new_ideal=ideal, new_nadir=nadir)
    
    print(f"✅ Created {name.upper()} with {n_var} vars and {n_obj} objectives.")
    return problem

## 5. Initialize ADM and Interactive Solvers

Define ADM parameters and initialize ADM

In [14]:
IT_LEARNING = 4
IT_DECISION = 3
N_VECTORS = 100


Run ADM-based interactive optimization for one problem.

In [ ]:
def run_adm_optimization(problem: Problem) -> pd.DataFrame:
    n_obj = len(problem.objectives)
    symbols = [f"{x.symbol}_min" for x in problem.objectives]
    reference_points = []

    # Initialize ADM
    adm = ADMAfsar(
        problem=problem,
        it_learning_phase=IT_LEARNING,
        it_decision_phase=IT_DECISION,
        number_of_vectors=N_VECTORS,
        true_ideal=problem.get_ideal_point(),
        true_nadir=problem.get_nadir_point(),
    )

    # Initial solvers
    solver_nsga3, publisher_nsga3 = nsga3(
        problem=problem,
        reference_vector_options={
            "reference_point": dict(zip(symbols, adm.preference)),
            "interactive_adaptation": "reference_point",
            "number_of_vectors": 100,
            })
    solver_rvea, publisher_rvea = rvea(
        problem=problem,
        reference_vector_options={ 
            "reference_point": dict(zip(symbols, adm.preference)),
            "interactive_adaptation": "reference_point",
            "number_of_vectors": 100,
            "creation_type": "multi-layer",
            "pymoo_layers": [{'strategy': 'das-dennis', 'n_partitions':12, 'scaling':1.0}]
        })

    archive_nsga3 = NonDominatedArchive(problem=problem, publisher=publisher_nsga3)
    archive_rvea = NonDominatedArchive(problem=problem, publisher=publisher_rvea)
    publisher_nsga3.auto_subscribe(archive_nsga3)
    publisher_rvea.auto_subscribe(archive_rvea)

    results_nsga3 = solver_nsga3()
    results_rvea = solver_rvea()

    iteration = 0
    while adm.has_next():
        iteration += 1
        front_rvea = results_rvea.outputs.select([f"f_{i+1}_min" for i in range(n_obj)]).to_numpy()
        front_nsga3 = results_nsga3.outputs.select([f"f_{i+1}_min" for i in range(n_obj)]).to_numpy()

        # Update ADM preference
        adm.get_next_preference(front_rvea, front_nsga3)
       
        # Determine phase
        phase = "L" if iteration <= IT_LEARNING else "D"

        # Store reference point with phase
        reference_points.append(list(adm.preference) + [phase])

        # Rebuild solvers with new reference point
        solver_nsga3, _ = nsga3(
            problem=problem,
            reference_vector_options={
                "reference_point": dict(zip(symbols, adm.preference)),
                "interactive_adaptation": "reference_point",
                "number_of_vectors": N_VECTORS,
            },
        )
        solver_rvea, _ = rvea(
            problem=problem,
            reference_vector_options={
                "reference_point": dict(zip(symbols, adm.preference)),
                "interactive_adaptation": "reference_point",
                "number_of_vectors": N_VECTORS,
            },
        )

        results_nsga3 = solver_nsga3()
        results_rvea = solver_rvea()

        phase = "Learning" if iteration <= IT_LEARNING else "Decision"

    # Convert to DataFrame
    columns = [f"f_{i+1}" for i in range(n_obj)] + ["Phase"]
    df_ref = pd.DataFrame(reference_points, columns=columns)
    return df_ref

# Example: Create a DTLZ2 problem with 5 objectives
problem = create_problem("dtlz2", 5)

reference_points = run_adm_optimization(problem)
print("\nGenerated Reference Points:")
print(reference_points)
    

✅ Created DTLZ2 with 14 vars and 5 objectives.

Generated Reference Points:
            f_1           f_2       f_3       f_4       f_5 Phase
0  1.104707e-19  1.090593e-19  0.000000  0.963659  0.321220     L
1  6.512699e-34  0.000000e+00  0.961352  0.000000  0.320451     L
2  2.408921e-34  4.145420e-01  0.000000  0.829084  0.414542     L
3  2.408921e-34  0.000000e+00  0.975282  0.325094  0.000000     L
4  5.028238e-01  0.000000e+00  0.502824  0.502824  0.502824     D
5  5.024864e-01  0.000000e+00  0.502486  0.502486  0.502486     D
6  5.017688e-01  0.000000e+00  0.501769  0.501769  0.501769     D


In [16]:
# # 6. Run All Experiments

dtlz_problems = [f"dtlz{i}" for i in range(1, 8)]
#wfg_problems = [f"wfg{i}" for i in range(1, 10)]
objective_counts = [3,5]

total = len(dtlz_problems ) * len(objective_counts)
counter = 0

start_time = time.time()
for name in dtlz_problems:
    for n_obj in objective_counts:
        counter += 1
        print(f"\n--- [{counter}/{total}] Running {name.upper()} with {n_obj} objectives ---")
        problem = create_problem(name, n_obj)
        reference_points = run_adm_optimization(problem)
        out_path = os.path.join(OUTPUT_DIR, f"{name}_{n_obj}obj_refpoints.csv")
        reference_points.to_csv(out_path, index=False)
        print(f"💾 Saved {len(reference_points)} reference points → {out_path}")

elapsed = (time.time() - start_time) / 60
print(f"\n✅ Completed all experiments in {elapsed:.2f} minutes.")



--- [1/14] Running DTLZ1 with 3 objectives ---
✅ Created DTLZ1 with 7 vars and 3 objectives.
name='dtlz1' description='Problem dtlz1 with 7 variables and 3 objectives.' constants=None variables=[Variable(name='x_1', symbol='x_1', variable_type=<VariableTypeEnum.real: 'real'>, lowerbound=0.0, upperbound=1.0, initial_value=None), Variable(name='x_2', symbol='x_2', variable_type=<VariableTypeEnum.real: 'real'>, lowerbound=0.0, upperbound=1.0, initial_value=None), Variable(name='x_3', symbol='x_3', variable_type=<VariableTypeEnum.real: 'real'>, lowerbound=0.0, upperbound=1.0, initial_value=None), Variable(name='x_4', symbol='x_4', variable_type=<VariableTypeEnum.real: 'real'>, lowerbound=0.0, upperbound=1.0, initial_value=None), Variable(name='x_5', symbol='x_5', variable_type=<VariableTypeEnum.real: 'real'>, lowerbound=0.0, upperbound=1.0, initial_value=None), Variable(name='x_6', symbol='x_6', variable_type=<VariableTypeEnum.real: 'real'>, lowerbound=0.0, upperbound=1.0, initial_value=N